# 07 — Semantic Fragmentation and Propagation

## Research Question
Do semantic fragmentation and propagation proxies reveal adversarial multi-agent behavior?

## Hypothesis
Raw fragmentation may reflect workflow complexity; risk-conditioned fragmentation should better isolate adversarial propagation.

## Input Data
- `episode_df_all`

## Prediction/Analysis Target
- Fragmentation, normalized fragmentation and adversarial fragmentation scores

## Validation Protocol
Score-based evaluation by seed; ROC-AUC/AUPRC; subgroup analysis by attack type and architecture.

## Expected Paper Artifact
- Fragmentation score table
- AUC/AUPRC analysis
- Propagation metric notes


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Propagation via semantic fragmentation

O trecho sobre fragmentar uma ação maliciosa em mensagens benignas pode virar um experimento forte:

A2A propagation can be detected by semantic drift across messages even when individual messages appear benign.

AKC-5 shows that semantic fragmentation is not sufficient by itself; it must be conditioned on risk-bearing signals such as suspicious tool use, compromised-agent influence, or instruction-injection markers.

semantic fragmentation proxy bruto = métrica de complexidade do workflow

colluding alto

Isso é plausível. Ataques de colusão podem produzir coordenação distribuída e múltiplas mensagens que parecem semanticamente alinhadas, mas convergem para uma decisão ruim.

IPI alto

Também plausível. IPI tende a espalhar instruções ou payloads por partes do trace.

contradicting alto

Plausível, porque contradição aumenta variação semântica e instabilidade.

DPI e impersonation baixos

Também faz sentido. DPI e impersonation podem ser ataques mais diretos, menos fragmentados: uma instrução explícita pode aparecer em um único ponto, sem grande decomposição semântica.

benign no topo

Não significa que benign é “mais adversarial”. Significa que tarefas benignas são multi-step e usam ferramentas diversas.

AFI = semantic drift × risk-bearing signal / functional complexity

* semantic drift: o quanto as mensagens divergem;

* risk-bearing signal: injeção, contradição, tool suspeita;

* functional complexity: número de tools, entropia de tools, número de mensagens.

Fragmentation alone is not adversarial. Fragmentation becomes adversarial when semantic drift is coupled with risk-bearing telemetry and cannot be explained by benign functional complexity.

In [ ]:
episode_fragmentation = episode_df_all.copy()
episode_fragmentation["adversarial_fragmentation_index"] = adversarial_fragmentation_index(episode_fragmentation)
episode_fragmentation["risk_conditioned_fragmentation"] = episode_fragmentation.apply(
    risk_conditioned_fragmentation,
    axis=1,
)
episode_fragmentation["semantic_fragmentation_proxy"] = episode_fragmentation.apply(
    semantic_fragmentation_proxy,
    axis=1,
)
episode_fragmentation = add_benign_normalized_fragmentation(episode_fragmentation)


# display(
#     episode_fragmentation
#     .groupby("attack_type")
#     .agg(
#         fragmentation_excess=("fragmentation_excess_over_benign", "mean"),
#         avg_tool_calls=("num_tool_calls", "mean"),
#         tool_entropy=("tool_call_entropy", "mean"),
#         raw_fragmentation=("semantic_fragmentation_proxy", "mean"),
#         adversarial_index=("adversarial_fragmentation_index", "mean"),
#         risk_index=("risk_conditioned_fragmentation", "mean")
#     )
#     .reset_index()
#     .sort_values("adversarial_index", ascending=False)
# )

In [ ]:
# if not episode_fragmentation.empty:
#     display_cols = [
#         "architecture", "model_name", "scenario", "attack_type", "is_attack",
#         "attack_success_proxy", "safe_refusal_proxy", "num_llm_calls",
#         "total_tokens", "latency_total_s", "num_tool_calls",
#         "attack_tool_invoked", "injection_marker_count",
#         "contradiction_marker_count", "avg_pairwise_message_similarity",
#     ]
#     display(episode_fragmentation[[c for c in display_cols if c in episode_fragmentation.columns]].head())

#     summary = episode_fragmentation.groupby(["architecture", "attack_type"], dropna=False).agg(
#         n=("task_id", "count"),
#         attack_rate=("is_attack", "mean"),
#         attack_success_proxy=("attack_success_proxy", "mean"),
#         safe_refusal_proxy=("safe_refusal_proxy", "mean"),
#         avg_tokens=("total_tokens", "mean"),
#         avg_latency_s=("latency_total_s", "mean"),
#         avg_tool_calls=("num_tool_calls", "mean"),
#         avg_injection_markers=("injection_marker_count", "mean"),
#         avg_contradiction_markers=("contradiction_marker_count", "mean"),
#     ).reset_index()
#     display(summary)
# else:
#     print("No features found.")


Fragmentação semântica bruta praticamente não detecta ataques; na verdade, ela está invertida, porque workflows benignos complexos têm mais fragmentação operacional. Mas quando a fragmentação é condicionada por sinais de risco, ela se torna altamente discriminativa.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y = episode_fragmentation["is_attack"].astype(int)
score = episode_fragmentation["adversarial_fragmentation_index"]

print("ROC-AUC:", roc_auc_score(y, score))
print("AUPRC:", average_precision_score(y, score))

In [ ]:
# score_eval_by_seed = evaluate_scores_by_seed(
#     episode_fragmentation,
#     score_cols=[
#         "semantic_fragmentation_proxy",
#         "fragmentation_excess_over_benign",
#         "adversarial_fragmentation_index",
#     ],
# )

# display(score_eval_by_seed)

# display(
#     score_eval_by_seed
#     .groupby("score")
#     .agg(
#         roc_auc_mean=("roc_auc", "mean"),
#         roc_auc_std=("roc_auc", "std"),
#         auprc_mean=("auprc", "mean"),
#         auprc_std=("auprc", "std"),
#     )
#     .reset_index()
#     .sort_values("roc_auc_mean", ascending=False)
# )

# print(episode_fragmentation["seed"].value_counts().sort_index())
# print(episode_fragmentation.shape)

In [ ]:
# from sklearn.metrics import roc_auc_score

# for score_col in [
#     "semantic_fragmentation_proxy",
#     "fragmentation_excess_over_benign",
#     "adversarial_fragmentation_index",
# ]:
#     print(
#         score_col,
#         roc_auc_score(
#             episode_fragmentation["is_attack"].astype(int),
#             episode_fragmentation[score_col].fillna(0),
#         )
#     )

9. O que eu priorizaria agora

A ordem que eu seguiria:

1. Cross-seed holdout para is_attack.
2. Ablation por grupos com leave-one-seed-out.
3. Early detection com 25/50/75/100% do trace.
4. Refinar fragmentação com risco e normalização contra benignos.
5. Criar benignos pareados automaticamente.
6. Substituir attack_success_proxy por outcomes comportamentais mais densos.
7. Adicionar uma segunda arquitetura, pelo menos single_agent_tamas, para mostrar que a telemetria não depende só de centralized_tamas.

Propagation Depth: número de agentes cujas mensagens incorporam conteúdo suspeito após o primeiro agente comprometido

Contagion Ratio: agentes afetados / total de agentes

Suspicious Influence: similaridade entre resposta final e mensagens do agente comprometido

Tool Propagation: ferramentas chamadas depois de uma mensagem suspeita

[ ] Benignos pareados por tarefa-base

[ ] 5 seeds ou pelo menos 3 com leave-one-seed-out

[ ] Attack-type classification incluindo benign

[ ] AKC/OODA phase prediction

[ ] AFI com ablação formal

[ ] Early detection por fração do trace

[ ] Leave-one-attack-out generalization

[ ] Compromised-agent localization

[ ] Telemetry-triggered intervention

[ ] Pelo menos 2 domínios - finance ou healthcare

[ ] Pelo menos 2 arquiteturas - decentralized_tamas/hybrid_tamas

Mostrar que a telemetria captura agentes comprometidos

Fazer leave-one-attack-out

Fazer validação leave-one-seed-out

Fazer ablação forte da métrica AFI

AFI full

AFI sem semantic drift

AFI sem risk signal

AFI sem functional normalization

AFI só tool features

AFI só linguistic markers

AFI só cost